In [2]:
import warnings
warnings.filterwarnings('ignore')
 
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, ConfusionMatrixDisplay
)
from sklearn.utils import resample

In [4]:
df = pd.read_csv('telecom.csv')
print("DATASET OVERVIEW")
print(f"Shape: {df.shape}")
print(f"Churn rate: {df['Churn'].mean()*100:.2f}%  (Imbalanced!)")
print(f"\nClass distribution:\n{df['Churn'].value_counts()}")
 

DATASET OVERVIEW
Shape: (667, 20)
Churn rate: 14.24%  (Imbalanced!)

Class distribution:
Churn
False    572
True      95
Name: count, dtype: int64


In [5]:

print("FEATURE ENGINEERING")

 
fe = df.copy()

fe['Total minutes']      = fe['Total day minutes'] + fe['Total eve minutes'] + \
                           fe['Total night minutes'] + fe['Total intl minutes']
fe['Total calls']        = fe['Total day calls'] + fe['Total eve calls'] + \
                           fe['Total night calls'] + fe['Total intl calls']
fe['Total charge']       = fe['Total day charge'] + fe['Total eve charge'] + \
                           fe['Total night charge'] + fe['Total intl charge']

#  Cost-per-minute 
fe['Charge per minute']  = fe['Total charge'] / (fe['Total minutes'] + 1e-6)
 
#  International usage ratio
fe['Intl usage ratio']   = fe['Total intl minutes'] / (fe['Total minutes'] + 1e-6)
 
#  Day usage dominance 
fe['Day usage ratio']    = fe['Total day minutes'] / (fe['Total minutes'] + 1e-6)
 
# 2e. Avg minutes per call
fe['Avg min per call']   = fe['Total minutes'] / (fe['Total calls'] + 1e-6)
 
# 2f. Customer service intensity flag (>3 calls = dissatisfied)
fe['High CS calls']      = (fe['Customer service calls'] > 3).astype(int)
 
# Voicemail engagement
fe['Vmail engagement']   = fe['Number vmail messages'] / \
                           (fe['Number vmail messages'].max() + 1e-6)
 
# Bin account length into tenure segments
fe['Tenure segment'] = pd.cut(fe['Account length'],
                               bins=[0, 50, 100, 150, 250],
                               labels=['New', 'Mid', 'Established', 'Loyal'])
 
print("Engineered features added:")
new_feats = ['Total minutes','Total calls','Total charge','Charge per minute',
             'Intl usage ratio','Day usage ratio','Avg min per call',
             'High CS calls','Vmail engagement','Tenure segment']
for f in new_feats:
    print(f"   {f}")

FEATURE ENGINEERING
Engineered features added:
   Total minutes
   Total calls
   Total charge
   Charge per minute
   Intl usage ratio
   Day usage ratio
   Avg min per call
   High CS calls
   Vmail engagement
   Tenure segment


In [6]:

# 3. PREPROCESSING

le = LabelEncoder()
 
# Binary encode Yes/No columns
fe['International plan']  = le.fit_transform(fe['International plan'])   # Yes=1
fe['Voice mail plan']     = le.fit_transform(fe['Voice mail plan'])       # Yes=1
 
# Drop State (high cardinality, low predictive power for small dataset)
# Drop Area code (essentially categorical with 3 values, low signal)
# Drop raw charge columns (multicollinear with Total charge)
drop_cols = ['State', 'Area code',
             'Total day charge','Total eve charge',
             'Total night charge','Total intl charge',
             'Tenure segment']  # already encoded as bin
fe = fe.drop(columns=drop_cols)
 
# Encode target
fe['Churn'] = fe['Churn'].astype(int)   # True→1, False→0
 
# Features / Target split
X = fe.drop(columns=['Churn'])
y = fe['Churn']
 
print(f"\nFinal feature set ({X.shape[1]} features):")
print(X.columns.tolist())
 


Final feature set (22 features):
['Account length', 'International plan', 'Voice mail plan', 'Number vmail messages', 'Total day minutes', 'Total day calls', 'Total eve minutes', 'Total eve calls', 'Total night minutes', 'Total night calls', 'Total intl minutes', 'Total intl calls', 'Customer service calls', 'Total minutes', 'Total calls', 'Total charge', 'Charge per minute', 'Intl usage ratio', 'Day usage ratio', 'Avg min per call', 'High CS calls', 'Vmail engagement']


In [17]:

# 4. TRAIN-TEST SPLIT (stratified)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
 
# logistic regression
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
 
print(f"\nTrain size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"Train churn rate: {y_train.mean()*100:.1f}%")
print(f"Test  churn rate: {y_test.mean()*100:.1f}%")
 



Train size: 533 | Test size: 134
Train churn rate: 14.3%
Test  churn rate: 14.2%


In [10]:
#handle class imablance 
X_train_df = pd.DataFrame(X_train_sc, columns=X.columns)
y_train_s  = y_train.reset_index(drop=True)
 
X_maj = X_train_df[y_train_s == 0]
X_min = X_train_df[y_train_s == 1]
y_maj = y_train_s[y_train_s == 0]
y_min = y_train_s[y_train_s == 1]
 
X_min_up, y_min_up = resample(X_min, y_min,
                               replace=True,
                               n_samples=len(X_maj),
                               random_state=42)
X_bal = pd.concat([X_maj, X_min_up])
y_bal = pd.concat([y_maj, y_min_up])
 
print(f"\nAfter oversampling: {y_bal.value_counts().to_dict()}")
 


After oversampling: {0: 457, 1: 457}


In [18]:

#  model training and evaluation
models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced', max_iter=1000, random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10, class_weight='balanced',
        random_state=42, n_jobs=-1
    )
}

In [19]:
 
results = {}

print("Model evaluation results")
for name, model in models.items():
    # Use balanced data for LR (already scaled), tree models use original balanced
    if name == 'Logistic Regression':
        model.fit(X_bal, y_bal)
        y_pred  = model.predict(X_test_sc)
        y_proba = model.predict_proba(X_test_sc)[:, 1]
    else:
        model.fit(X_bal, y_bal)
        y_pred  = model.predict(X_test_sc)
        y_proba = model.predict_proba(X_test_sc)[:, 1]
 
    roc_auc = roc_auc_score(y_test, y_proba)
    report  = classification_report(y_test, y_pred, output_dict=True)
    cm      = confusion_matrix(y_test, y_pred)
 
    results[name] = {
        'model': model,
        'y_pred': y_pred,
        'y_proba': y_proba,
        'roc_auc': roc_auc,
        'report': report,
        'cm': cm,
    }
 
    print(f"  {name}") 
    print(f"  Accuracy : {report['accuracy']:.4f}")
    print(f"  Precision (Churn): {report['1']['precision']:.4f}")
    print(f"  Recall    (Churn): {report['1']['recall']:.4f}")
    print(f"  F1-Score  (Churn): {report['1']['f1-score']:.4f}")
    print(f"  ROC-AUC  : {roc_auc:.4f}")
 

Model evaluation results
  Logistic Regression
  Accuracy : 0.8881
  Precision (Churn): 0.5714
  Recall    (Churn): 0.8421
  F1-Score  (Churn): 0.6809
  ROC-AUC  : 0.8778
  Random Forest
  Accuracy : 0.9701
  Precision (Churn): 1.0000
  Recall    (Churn): 0.7895
  F1-Score  (Churn): 0.8824
  ROC-AUC  : 0.9259
